#Sistema di raccomandazione
Il sistema implementa un approccio di raccomandazione one-time, evitando la costruzione di matrici di similarità NxN. Applica un filtro hard sulla categoria e combina una similarità basata su ingredienti con una funzione deterministica di brand e segmento, pesate rispettivamente 0.8 e 0.2.



In [26]:
import pandas as pd
import numpy as np



1) Importo il df e creo un nuovo df con le colonne d'interesse



In [27]:
df = pd.read_csv("brand_categoria.csv",dtype={"code": str})

df = df[['code', 'product_name','brand_name', 'brand_segment','macro_category']].copy()
df['code'] = df['code'].astype(str)
df['product_name'] = df['product_name'].astype(str)
df['brand_name'] = df['brand_name'].astype(str)
df['brand_segment'] = df['brand_segment'].astype(str)
df['macro_category'] = df['macro_category'].astype(str)


df.tail(30)



,code,product_name,brand_name,brand_segment,macro_category
31026,5901887010371,eye make up remover,ziaja,middle,Makeup
31027,5901887010173,crema facial pepino,ziaja,middle,Face_Care
31028,5901887001348,gel limpiador de aceite de oliva,ziaja,middle,Oils
31029,5901887016953,crema facial ultra ligera aceite de oliva,ziaja,middle,Face_Care
31030,5901887008460,hand cream,ziaja,middle,Face_Care
31031,5901887006916,sensitive soothing day cream,ziaja,middle,Face_Care
31032,3700436313589,bush magic,zingus,mass_market,Other
31033,7596211001033,crema hidratante dia fps 15,zoah,mass_market,Face_Care
31034,7596211000203,crema corporal para la celulitis,zoah,mass_market,Body_Care
31035,4770161103723,balksvuju gyslociu luobeles,zolynelis,mass_market,Other


In [28]:
df_ing = pd.read_csv("ingr_cat.csv",dtype={"code": str})


df_ing['code'] = df_ing['code'].astype(str)
df_ing['product_name'] = df_ing['product_name'].astype(str)
df_ing['macro_category'] = df_ing['macro_category'].astype(str)
df_ing['inci_name'] = df_ing['inci_name'].astype(str)

df_finale = (
    df_ing
    .groupby(['code', 'product_name', 'macro_category'])['inci_name']
    .apply(lambda lst: ", ".join(sorted(set(lst))))
    .reset_index()
    .rename(columns={'inci_name': 'ingredients'}))

df_finale.tail(20)


,code,product_name,macro_category,ingredients
15111,96121214,Powder soft anti perspirant,Other,"Alpha-Isomethyl Lonone, Aluminum Zirconium Tet..."
15112,96125694,REXONA Déodorant Homme Bille Anti Transpirant ...,Deodorants,"Alpha-Isomethyl Ionone, Aluminum Zirconium Tet..."
15113,96125748,Motionsense active shield,Other,"Alpha-Lsomethyl Lonone, Aluminum Chlorohydrate..."
15114,96130339,Dove zero original 50ml,Body_Care,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Citric..."
15115,96132289,Axe Déodorant Anti Bactérien You Spray 100ml,Deodorants,"Alcohol Denat, Alpha-Isomethyl Ionone, Butane,..."
15116,96132876,Monsavon Déodorant Femme Spray Compressé Fleur...,Hygiene,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Bht, B..."
15117,96132906,Monsavon Déodorant Femme Spray Compressé Grena...,Hygiene,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Bht, B..."
15118,96134696,DOVE Déodorant Femme Spray Compressé Pierre d'...,Body_Care,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Benzyl..."
15119,96148402,Monsavon Déodorant Femme Spray Compressé Vanil...,Hygiene,"Benzyl Alcohol, Bht, Butane, Butyrospermum Par..."
15120,96183281,sensodyne,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Die 7..."


In [29]:
#per la semplificazione della ricerca in fase di test, è una lista di "code" in comune tra i dataset
common_codes = set(df['code']).intersection(set(df_finale['code']))

print(f"Prodotti in df: {len(df)}")
print(f"Prodotti in df_finale: {len(df_finale)}")
print(f"Prodotti comuni: {len(common_codes)}")
common_codes

Prodotti in df: 31056
Prodotti in df_finale: 15131
Prodotti comuni: 12304


{'1237654876548',
 '0735303504312',
 '8431876282177',
 '3593290031993',
 '3596209937119',
 '3380390900904',
 '4337256573030',
 '3600541072992',
 '7790740508401',
 '0651986800216',
 '3596710524129',
 '8712561535243',
 '8004020931766',
 '4311501066409',
 '3250391571642',
 '8710447365724',
 '7501026011511',
 '3282770154559',
 '3282779363815',
 '8710522378564',
 '4311596019427',
 '8710447349960',
 '42345060',
 '3489940081084',
 '3517360018288',
 '3251248046047',
 '3574661091020',
 '8851515605974',
 '3468080145047',
 '3600523279319',
 '3600541540262',
 '3181730125007',
 '0764302290629',
 '3549620007602',
 '3574661002378',
 '3700992600048',
 '4005900035622',
 '3760177361607',
 '3517360010572',
 '3760179070262',
 '0810112515848',
 '0761571362200',
 '5022496101301',
 '3600550284256',
 '3245678562277',
 '7899706192712',
 '4066447106275',
 '4010355296306',
 '7046110066775',
 '20527310',
 '7702018366200',
 '4005808484843',
 '8480017839824',
 '3574661464978',
 '0769915152203',
 '3661596986436',
 '

**2. Assegnazione pesi ai brand_segment.**

Il sistema di suggerimento tiene conto del segmento di mercato e pesa la similarità in base ad esso.
Prodotti dello stesso segmento avranno peso maggiore.

In [30]:
SEGMENT_SIMILARITY = {
    ('middle', 'middle'): 0.7,
    ('mass_market', 'mass_market'): 0.7,
    ('luxury', 'luxury'): 0.7,

    ('mass_market', 'middle'): 0.5,
    ('middle', 'mass_market'): 0.5,

    ('mass_market', 'luxury'): 0.3,
    ('luxury', 'mass_market'): 0.3,

    ('middle', 'luxury'): 0.5,
    ('luxury', 'middle'): 0.5
}


**3. Creazione dell'array di similarità 1xK.**

Per il brand non uso una cosine similarity, ma una funzione deterministica basata su regole di business che tengono conto sia dell’identità del brand sia del segmento di mercato.

Il sistema applica un hard filter sulla categoria per ridurre lo spazio di ricerca e garantire coerenza semantica, e successivamente calcola una similarità pesata basata su ingredienti e su regole di brand e segmento.



In [31]:
def brand_segment_similarity_1xK(
    query_product_id,
    df,
    category_col='macro_category',
    segment_col='brand_segment'
):
    query_product_id = str(query_product_id)

    if query_product_id not in df['code'].values:
        raise ValueError("Codice prodotto non trovato")

    query_row = df[df['code'] == query_product_id].iloc[0]
    query_category = query_row[category_col]
    query_brand = query_row['brand_name']
    query_segment = query_row[segment_col]
    query_product_name = query_row['product_name']  # ← AGGIUNTA

    # Filtro hard: stessa categoria -> ricerca tra prodotti della stessa categoria
    df_filtered = df[df[category_col] == query_category].copy()
    df_filtered = df_filtered[df_filtered['code'] != query_product_id]
    #filtro per evitare che restituisca lo stesso prodotto
    df_filtered['query_brand_name'] = query_brand
    df_filtered['query_product_name'] = query_product_name

    similarities = []

    for _, row in df_filtered.iterrows():
        if row['brand_name'] == query_brand:
            similarities.append(1.0)#se il prodotto appartiene allo stesso brand avrà similarità massima
        else:
            pair = (query_segment, row[segment_col]) #altrimenti considera il segmento di mercato
            similarities.append(SEGMENT_SIMILARITY.get(pair, 0.0))
    df_filtered['similarity'] = similarities
    return np.array(similarities), df_filtered




4. Applicazione della funzione e creazione dell'aray one-time a partire dal prodotto


In [32]:
query_id = input('Inserisci il codice del prodotto: ')

brand_sim = brand_segment_similarity_1xK(query_id, df)
#non ottengo una matrice di similarità NXN, ma un array 1xK:
#cioè la similarità del prodotto query rispetto a tutti gli altri prodotti della stessa categoria



#brand_sim è una tupla: il primo elemento è l'array 1xK, il secondo elemento è il dataframe con i prodotti candidati
#brand_sim == (similarity_array, df_filtered).
#la tupla serve a sapere a quali prodotti mi riferisco
brand_sim_array, df_brand_candidates = brand_sim



In [33]:
brand_sim_array[:10] #vedo i primi dieci valori di similarità

array([0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7])

In [34]:
df_brand_candidates = df_brand_candidates[[
    'code',
    'query_product_name',
    'query_brand_name',
    'product_name',
    'brand_name',
    'brand_segment',
    'macro_category',
    'similarity'
]]


In [35]:
df_brand_candidates.sort_values('similarity', ascending= False).head(10) #vedo i primi dieci candidati


,code,query_product_name,query_brand_name,product_name,brand_name,brand_segment,macro_category,similarity
26172,5054563256522,sensodyne repair protect,sensodyne,sensodyne expert protect rapide action,sensodyne,mass_market,Other,1.0
26174,8710464140021,sensodyne repair protect,sensodyne,pro glasur,sensodyne,mass_market,Other,1.0
26158,7498100701454,sensodyne repair protect,sensodyne,repare et protege,sensodyne,mass_market,Other,1.0
26159,5054563185815,sensodyne repair protect,sensodyne,fresh mint,sensodyne,mass_market,Other,1.0
26160,5054563155658,sensodyne repair protect,sensodyne,repair protect,sensodyne,mass_market,Other,1.0
26162,5054563186249,sensodyne repair protect,sensodyne,zahnpasta,sensodyne,mass_market,Other,1.0
26165,5054563156594,sensodyne repair protect,sensodyne,rapid zahnpasta,sensodyne,mass_market,Other,1.0
26168,5054563206688,sensodyne repair protect,sensodyne,sensodyne soin gencives,sensodyne,mass_market,Other,1.0
26169,5054563209290,sensodyne repair protect,sensodyne,sensodyne proschmelz,sensodyne,mass_market,Other,1.0
26170,5054563210180,sensodyne repair protect,sensodyne,zahnpasta clinical repair,sensodyne,mass_market,Other,1.0


5. Con questa funzione si trovano i prodotti più simili a un prodotto dato, guardando solo gli ingredienti. Prima limita il confronto ai prodotti della stessa categoria, così il confronto ha senso.
Poi confronta gli ingredienti del prodotto scelto con quelli degli altri e dice quanto si assomigliano, usando un punteggio numerico.

In [36]:
#pulizia per esclusione ingredienti che potrebbero compromettere il corretto
#calcolo della similarità
EXCLUDE_INGREDIENTS = {
    'aqua', 'water', 'parfum', 'fragrance',
    'ci', 'color', 'colour',
    'sodium chloride'
}

def clean_ingredients(ing_string):
    ing = [
        i.strip().lower()
        for i in ing_string.split(',')
        if len(i.strip()) > 2
    ]
    ing = [
        i for i in ing
        if not any(x in i for x in EXCLUDE_INGREDIENTS)
        and not i.startswith('ci ')
    ]
    return list(set(ing))

In [37]:
df_finale['ingredients_list'] = df_finale['ingredients'].apply(clean_ingredients)

df_finale[['ingredients', 'ingredients_list']].head(10)

,ingredients,ingredients_list
0,"Arnica Montana, Avoid Contact With Eyes, Burit...","[tocopherol, buriti oil, menthol, silicones, a..."
1,Oil,[oil]
2,"Added Sugarss, Colorings, Cr 6Ad, Croydon, Lac...","[lactose, cr 6ad, croydon, preservatives, adde..."
3,Oil,[oil]
4,"Lilium Candidum Flower Extract, Oil","[lilium candidum flower extract, oil]"
5,"Allantoin, Aloe Barbadensis Leaf Juice, Benzoi...","[allantoin, glycerin, aloe barbadensis leaf ju..."
6,"Coco-Glucoside, Decyl Glucoside, Glycerin, Gly...","[leuconostocradish root ferment filtrate, glyc..."
7,Lawsonia Inermis Leaf Powder,[lawsonia inermis leaf powder]
8,Natural Calcium Bentonite Clay,[]
9,"Cellulose Gum, Citric Acid, Cocamidopropyl Bet...","[sodium saccharin, glycerin, titanium dioxide,..."


In [38]:
def jaccard_similarity(set_a, set_b):
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

In [39]:
def ingredient_similarity_1xK(
    query_product_id,
    df_finale,
    category_col='macro_category'
):
    query_product_id = str(query_product_id)

    if query_product_id not in df_finale['code'].values:
        raise ValueError("Codice prodotto non trovato")

    query_row = df_finale[df_finale['code'] == query_product_id].iloc[0]
    query_category = query_row[category_col]
    query_ing = set(query_row['ingredients_list'])

    # Filtro hard: stessa categoria
    df_filtered = df_finale[df_finale[category_col] == query_category].copy()
    df_filtered = df_filtered[df_filtered['code'] != query_product_id]



    print(f"Trovati {len(df_filtered)} prodotti nella stessa categoria")

    similarities = []

    for _, row in df_filtered.iterrows():
        sim = jaccard_similarity(
            query_ing,
            set(row['ingredients_list'])
        )
        similarities.append(sim)

    return np.array(similarities), df_filtered



In [40]:
ing_sim_array, df_ing_candidates = ingredient_similarity_1xK(query_id, df_finale)

ing_sim_array[:10]

Trovati 5582 prodotti nella stessa categoria


array([0.        , 0.        , 0.0625    , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.04166667, 0.05263158])

In [41]:
#aggiunta similarità e is_query per vedere quali sono i prod raccomandati e
#qual è il prodotto originale cercato
df_ing_candidates_temp = df_ing_candidates[df_ing_candidates['code'] != query_id].copy()
df_ing_candidates_temp['ing_sim'] = ing_sim_array
df_ing_candidates_temp['is_query'] = False
df_ing_candidates_temp = df_ing_candidates_temp.sort_values(by='ing_sim', ascending=False).reset_index(drop=True)


df_query_ing = df_finale[df_finale['code'] == query_id].copy()
df_query_ing['ing_sim'] = 1.0        # similarità massima con se stesso
df_query_ing['is_query'] = True

df_ing_candidates = pd.concat(
    [df_query_ing, df_ing_candidates_temp],
    ignore_index=True
)

df_ing_candidates.head(10)



,code,product_name,macro_category,ingredients,ingredients_list,ing_sim,is_query
0,8901571010844,Sensodyne Repair & Protect,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",1.000000,True
1,3094905000033,Répare & Protège,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.916667,False
2,5054563155658,Repair & Protect,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.916667,False
3,5054563006592,Sensodyne repair & protect,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.916667,False
4,6001076192525,tooth,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, contains sodium fluo...",0.846154,False
5,5054563109743,Sensodyne sensibilidade e gengivas,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.733333,False
6,5054563010278,Rapide Action Extra Fresh,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.666667,False
7,05050499,Sensodyne Sensibilité & Gencives,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.666667,False
8,5054563024558,Rapide Action,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.600000,False
9,3094904751622,Rapide action,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.600000,False


In [42]:
query_product_ingredients = set(
    df_finale[df_finale['code'] == query_id]['ingredients_list'].iloc[0]
)

def get_common_ingredients(candidate_ingredients):
    return list(query_product_ingredients.intersection(set(candidate_ingredients)))

df_ing_candidates['common_ingr'] = df_ing_candidates['ingredients_list'].apply(get_common_ingredients)

df_ing_candidates[['code', 'product_name', 'ingredients_list', 'common_ingr']].head(10)

,code,product_name,ingredients_list,common_ingr
0,8901571010844,Sensodyne Repair & Protect,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
1,3094905000033,Répare & Protège,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
2,5054563155658,Repair & Protect,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
3,5054563006592,Sensodyne repair & protect,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
4,6001076192525,tooth,"[aroma, sodium saccharin, contains sodium fluo...","[aroma, sodium saccharin, carbomer, glycerin, ..."
5,5054563109743,Sensodyne sensibilidade e gengivas,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
6,5054563010278,Rapide Action Extra Fresh,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
7,05050499,Sensodyne Sensibilité & Gencives,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
8,5054563024558,Rapide Action,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."
9,3094904751622,Rapide action,"[aroma, sodium saccharin, carbomer, glycerin, ...","[aroma, sodium saccharin, carbomer, glycerin, ..."


In [43]:
ing_sim_array[:10]

array([0.        , 0.        , 0.0625    , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.04166667, 0.05263158])

In [44]:
df_ing_candidates.head(10)

,code,product_name,macro_category,ingredients,ingredients_list,ing_sim,is_query,common_ingr
0,8901571010844,Sensodyne Repair & Protect,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",1.000000,True,"[aroma, sodium saccharin, carbomer, glycerin, ..."
1,3094905000033,Répare & Protège,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.916667,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
2,5054563155658,Repair & Protect,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.916667,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
3,5054563006592,Sensodyne repair & protect,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.916667,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
4,6001076192525,tooth,Other,"Aroma, Calcium Sodium Phosphosilicate, Carbome...","[aroma, sodium saccharin, contains sodium fluo...",0.846154,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
5,5054563109743,Sensodyne sensibilidade e gengivas,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.733333,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
6,5054563010278,Rapide Action Extra Fresh,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.666667,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
7,05050499,Sensodyne Sensibilité & Gencives,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.666667,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
8,5054563024558,Rapide Action,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.600000,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."
9,3094904751622,Rapide action,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Glyce...","[aroma, sodium saccharin, carbomer, glycerin, ...",0.600000,False,"[aroma, sodium saccharin, carbomer, glycerin, ..."


**5. Indicizzazione su codice prodotto**

Costruzione nuovo DF: unione dei due dataframe sul codice prodotto al fine del calcolo della similarità finale.

In [45]:
df_ing= df_ing_candidates[['code', 'ing_sim']].copy()

df_brand = df_brand_candidates[['code']].copy()
df_brand['brand_sim'] = brand_sim_array

#allineo la similarità
df_merged = df_ing.merge(
    df_brand,
    on='code',
    how='inner'
)


In [46]:
#check per verificare che l'allineamento è corretto
assert len(df_merged) > 0
assert df_merged['code'].is_unique

df_merged.head()


,code,ing_sim,brand_sim
0,3094905000033,0.916667,0.7
1,5054563155658,0.916667,1.0
2,5054563006592,0.916667,1.0
3,5054563109743,0.733333,1.0
4,5054563010278,0.666667,1.0


**6. Calcolo della similarità finale**

Gli ingredienti hanno peso maggiore nel determinare il prodotto più simile (0.8). I brand hanno peso minore (0.2).


In [47]:
df_merged['final_similarity'] = (
    0.8 * df_merged['ing_sim'] +
    0.2 * df_merged['brand_sim']
)


**6. Costruzione dataframe per la visualizzazione del prodotto più simile**


In [49]:
query_product_name = (
    df.loc[df['code'] == query_id, 'product_name']
    .iloc[0]
)

ingredients_query = (
    df_ing_candidates['ingredients']
    .iloc[0]
)


risultato = pd.DataFrame({
    'query_product_id': query_id,
    'query_product_name': query_product_name,
    'recommended_product_id': df_merged['code'],
    'final_similarity_score': df_merged['final_similarity'],
    'common_ingredients': df_ing_candidates['common_ingr'],
    'ingredients_query': ingredients_query

})

#ordino i prodotti per suggerire il prodotto più simile
risultato = risultato.sort_values(
    by='final_similarity_score',
    ascending=False
).reset_index(drop=True)

risultato['rank'] = risultato.index + 1

# arricchimento informativo (nome, brand, segmento)
risultato = risultato.merge(
    df[['code', 'product_name', 'brand_name', 'brand_segment']],
    left_on='recommended_product_id',
    right_on='code',
    how='left'
).drop(columns='code')

risultato['num_common_ingredients'] = risultato['common_ingredients'].apply(
    lambda x: len(x) if isinstance(x, (list, set, tuple)) else len(str(x).split(','))
)

risultato = risultato[
    [
        'rank',
        'query_product_id',
        'query_product_name',
        'ingredients_query',
        'recommended_product_id',
        'product_name',
        'brand_name',
        'brand_segment',
        'common_ingredients',
        'num_common_ingredients',
        'final_similarity_score'
    ]
]



risultato.head(10)



,rank,query_product_id,query_product_name,ingredients_query,recommended_product_id,product_name,brand_name,brand_segment,common_ingredients,num_common_ingredients,final_similarity_score
0,1,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563006592,sensodyne repair protect,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",11,0.933333
1,2,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563155658,repair protect,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",11,0.933333
2,3,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",3094905000033,repare protege,glaxosmithkline,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",12,0.873333
3,4,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563109743,sensodyne sensibilidade e gengivas,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",11,0.786667
4,5,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563010278,rapide action extra fresh,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",11,0.733333
5,6,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563024558,rapide action,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",10,0.680000
6,7,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",3094904751622,rapide action,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",10,0.680000
7,8,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",05050499,sensodyne sensibilite gencives,gsk,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",11,0.673333
8,9,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563049957,sensibilite et gencives,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",9,0.650000
9,10,8901571010844,sensodyne repair protect,"Aroma, Calcium Sodium Phosphosilicate, Carbome...",5054563256522,sensodyne expert protect rapide action,sensodyne,mass_market,"[aroma, sodium saccharin, carbomer, glycerin, ...",9,0.650000
